In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [6]:
import numpy as np
import pandas as pd

images = np.load("../data/images_plant.npy")

labels_df = pd.read_csv("../data/Labels_plant.csv")

labels = labels_df["Label"]

print("Image shape:", images.shape)
print(labels_df.head())

Image shape: (4750, 128, 128, 3)
                       Label
0  Small-flowered Cranesbill
1  Small-flowered Cranesbill
2  Small-flowered Cranesbill
3  Small-flowered Cranesbill
4  Small-flowered Cranesbill


In [7]:
print("Total images:", len(images))
print("Total labels:", len(labels))

Total images: 4750
Total labels: 4750


In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(labels)

print(le.classes_)

['Black-grass' 'Charlock' 'Cleavers' 'Common Chickweed' 'Common wheat'
 'Fat Hen' 'Loose Silky-bent' 'Maize' 'Scentless Mayweed'
 'Shepherds Purse' 'Small-flowered Cranesbill' 'Sugar beet']


In [9]:
X = images / 255.0

In [10]:
print(X.shape)

(4750, 128, 128, 3)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [12]:
print(X_train.shape)
print(X_test.shape)

(3800, 128, 128, 3)
(950, 128, 128, 3)


In [13]:
import tensorflow as tf

# Recreate the same architecture
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(12,activation='softmax')
])

d:\plant-seedling-classifier\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,306,188 (12.61 MB)

 Trainable params: 3,306,188 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=32
)

Epoch 1/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 26s 241ms/step - accuracy: 0.2467 - loss: 2.2159 - val_accuracy: 0.4079 - val_loss: 1.7862
Epoch 2/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 232ms/step - accuracy: 0.4332 - loss: 1.6316 - val_accuracy: 0.6000 - val_loss: 1.2188
Epoch 3/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 231ms/step - accuracy: 0.5299 - loss: 1.3589 - val_accuracy: 0.6079 - val_loss: 1.1292
Epoch 4/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 234ms/step - accuracy: 0.5895 - loss: 1.2155 - val_accuracy: 0.6842 - val_loss: 0.9713
Epoch 5/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 236ms/step - accuracy: 0.6339 - loss: 1.0827 - val_accuracy: 0.7079 - val_loss: 0.9175
Epoch 6/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 236ms/step - accuracy: 0.6536 - loss: 0.9804 - val_accuracy: 0.7316 - val_loss: 0.7740
Epoch 7/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 236ms/step - accuracy: 0.6826 - loss: 0.9127 - val_accuracy: 0.7605 - val_loss: 0.7182
Epoch 8/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 22s 234ms/step - accuracy: 0.7155 - loss: 0.8025 - val_accu

In [17]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.7684 - loss: 0.9228
Test Loss: 0.9228408336639404
Test Accuracy: 0.7684210538864136


In [18]:
train_accuracy = history.history['accuracy'][-1]
val_accuracy = history.history['val_accuracy'][-1]

train_loss = history.history['loss'][-1]
val_loss = history.history['val_loss'][-1]

print("Final Training Accuracy:", train_accuracy)
print("Final Validation Accuracy:", val_accuracy)

print("Final Training Loss:", train_loss)
print("Final Validation Loss:", val_loss)

Final Training Accuracy: 0.8674342036247253
Final Validation Accuracy: 0.7921052575111389
Final Training Loss: 0.34276556968688965
Final Validation Loss: 0.8007349967956543


In [19]:
metrics = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Accuracy": [train_accuracy, val_accuracy, test_accuracy],
    "Loss": [train_loss, val_loss, test_loss]
})

metrics

,Dataset,Accuracy,Loss
0,Training,0.867434,0.342766
1,Validation,0.792105,0.800735
2,Test,0.768421,0.922841


In [20]:
import numpy as np

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step


In [21]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
)

print(report)

                           precision    recall  f1-score   support

              Black-grass       0.38      0.34      0.36        53
                 Charlock       0.85      0.95      0.90        78
                 Cleavers       0.81      0.72      0.76        58
         Common Chickweed       0.85      0.93      0.89       122
             Common wheat       0.82      0.52      0.64        44
                  Fat Hen       0.85      0.78      0.81        95
         Loose Silky-bent       0.72      0.82      0.77       131
                    Maize       0.80      0.80      0.80        44
        Scentless Mayweed       0.69      0.77      0.72       103
          Shepherds Purse       0.60      0.59      0.59        46
Small-flowered Cranesbill       0.84      0.85      0.84        99
               Sugar beet       0.87      0.69      0.77        77

                 accuracy                           0.77       950
                macro avg       0.76      0.73      0.74    

In [22]:
model.load_weights("../models/plant_classifier.keras")

In [23]:
model.save("../models/plant_classifier.keras", save_format="keras")

In [24]:
model.save_weights("../models/plant_weights.weights.h5")